# Phase 2b: Conditional GAN (cGAN) — BraTS 2025 [FIXED v2]

**Changes in v2:**
- ✅ Cell 6 stuck fix — tqdm flush + sys.stdout, per-batch print every N batches
- ✅ A600 GPU batch sizes: 256 / 128 / 64 per stage
- ✅ Real image input (not noise)
- ✅ Spectral Normalization on Discriminator
- ✅ Rebalanced loss: LAM_L1=10, LAM_SSIM=5
- ✅ Full 4-channel composite ROI mask
- ✅ Cosine LR annealing within each stage
- ✅ Label smoothing
- ✅ Gradient clipping

In [19]:
# ══════════════ Cell 1: Imports & Device ══════════════
import os, sys, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import spectral_norm
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    from pytorch_msssim import ssim as ssim_fn
    USE_SSIM = True
    print("✓ pytorch_msssim found")
except ImportError:
    USE_SSIM = False
    print("⚠ pytorch_msssim not found — pip install pytorch-msssim")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("CPU mode")
torch.backends.cudnn.benchmark = True

✓ pytorch_msssim found
✓ GPU: NVIDIA RTX A6000
  VRAM: 51.5 GB


In [ ]:
# ══════════════ Cell 2: Dataset ══════════════
PROC_DIR = "/kaggle/working/processed"
MOD = 1

class BraTSDataset(Dataset):
    def __init__(self, split="train", mod=1):
        self.sd    = Path(PROC_DIR) / split / "slices"
        self.md    = Path(PROC_DIR) / split / "masks"
        self.files = sorted(self.sd.glob("*.npy"))
        self.mod   = mod
        print(f"  {split}: {len(self.files)} slices")
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        f   = self.files[i]
        img = torch.from_numpy(np.load(str(f)).astype(np.float32)[self.mod:self.mod+1])
        msk = torch.from_numpy(np.load(str(self.md / f.name)).astype(np.float32))
        return img, msk

train_ds = BraTSDataset("train", MOD)
val_ds   = BraTSDataset("val",   MOD)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

NameError: name 'Dataset' is not defined

In [21]:
# ══════════════ Cell 3: Shared Blocks & Loss Functions ══════════════

class DepthwiseSeparableConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, stride=1, pad=1, bias=False):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, k, stride, pad, groups=in_ch, bias=bias)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=bias)
    def forward(self, x): return self.pw(self.dw(x))

class DSConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, stride=1, act="leaky_relu", bn=True):
        super().__init__()
        layers = [DepthwiseSeparableConv2d(in_ch, out_ch, k, stride, k // 2)]
        if bn: layers.append(nn.BatchNorm2d(out_ch))
        layers.append({"relu": nn.ReLU(True), "leaky_relu": nn.LeakyReLU(0.2, True), "none": nn.Identity()}.get(act, nn.LeakyReLU(0.2, True)))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

def bce_loss(pred, is_real, smooth=True):
    target = torch.full_like(pred, 0.9 if (is_real and smooth) else (0.1 if (not is_real and smooth) else float(is_real)))
    return F.binary_cross_entropy_with_logits(pred, target)

def composite_roi(masks):
    return masks.max(dim=1, keepdim=True).values

def roi_l1(pred, target, masks, w=5.0):
    roi    = composite_roi(masks)
    weight = torch.ones_like(roi) + roi * (w - 1)
    return (weight * torch.abs(pred - target)).mean()

def ssim_loss(pred, target):
    if USE_SSIM:
        return 1.0 - ssim_fn(pred.clamp(0,1), target.clamp(0,1), data_range=1.0, size_average=True)
    return torch.tensor(0.0, device=pred.device)

print("✓ Blocks & losses ready")

✓ Blocks & losses ready


In [22]:
# ══════════════ Cell 4: Progressive Scheduler (A600 batch sizes) ══════════════

class PSched:
    # A600 has enough VRAM for large batches
    STAGES = [
        {"res": 32,  "epochs": 40,  "lr": 2e-4, "batch": 512},
        {"res": 64,  "epochs": 70,  "lr": 1e-4, "batch": 512},
        {"res": 128, "epochs": 90, "lr": 5e-5, "batch": 512},
    ]
    def __init__(self):              self.i = 0
    @property
    def stage(self):                 return self.STAGES[self.i]
    @property
    def res(self):                   return self.stage["res"]
    @property
    def epochs(self):                return self.stage["epochs"]
    @property
    def lr(self):                    return self.stage["lr"]
    @property
    def batch(self):                 return self.stage["batch"]
    @property
    def n(self):                     return len(self.STAGES)
    def advance(self):
        if self.i < len(self.STAGES) - 1: self.i += 1; return True
        return False
    def resize(self, x):
        r = self.res
        return F.interpolate(x, size=(r,r), mode="bilinear", align_corners=False) if x.shape[-1] != r else x
    def set_lr(self, *opts):
        for o in opts:
            for pg in o.param_groups: pg["lr"] = self.lr
    def __repr__(self):
        return f"Stage {self.i+1}/{len(self.STAGES)} | res={self.res} lr={self.lr:.1e} batch={self.batch}"

print("✓ Scheduler ready")

✓ Scheduler ready


In [23]:
# ══════════════ Cell 5: Architecture ══════════════

class cGANGenerator(nn.Module):
    def __init__(self, in_ch=1, cond_ch=4, out_ch=1, bf=32):
        super().__init__()
        ti = in_ch + cond_ch
        self.enc1 = nn.Sequential(DSConvBlock(ti,   bf),   DSConvBlock(bf,   bf),   nn.MaxPool2d(2))
        self.enc2 = nn.Sequential(DSConvBlock(bf,   bf*2), DSConvBlock(bf*2, bf*2), nn.MaxPool2d(2))
        self.enc3 = nn.Sequential(DSConvBlock(bf*2, bf*4), DSConvBlock(bf*4, bf*4), nn.MaxPool2d(2))
        self.enc4 = nn.Sequential(DSConvBlock(bf*4, bf*8), DSConvBlock(bf*8, bf*8), nn.MaxPool2d(2))
        self.bot  = nn.Sequential(DSConvBlock(bf*8, bf*8), DSConvBlock(bf*8, bf*8))
        self.u4 = nn.ConvTranspose2d(bf*8, bf*4, 2, stride=2)
        self.d4 = nn.Sequential(DSConvBlock(bf*8, bf*4), DSConvBlock(bf*4, bf*4))
        self.u3 = nn.ConvTranspose2d(bf*4, bf*2, 2, stride=2)
        self.d3 = nn.Sequential(DSConvBlock(bf*4, bf*2), DSConvBlock(bf*2, bf*2))
        self.u2 = nn.ConvTranspose2d(bf*2, bf,   2, stride=2)
        self.d2 = nn.Sequential(DSConvBlock(bf*2, bf),   DSConvBlock(bf,   bf))
        self.u1 = nn.ConvTranspose2d(bf,   bf,   2, stride=2)
        self.d1 = nn.Sequential(DSConvBlock(bf+ti, bf),  DSConvBlock(bf,   bf))
        self.out = nn.Sequential(nn.Conv2d(bf, out_ch, 1), nn.Sigmoid())
    def forward(self, x, cond):
        inp = torch.cat([x, cond], 1)
        e1=self.enc1(inp); e2=self.enc2(e1); e3=self.enc3(e2); e4=self.enc4(e3)
        b=self.bot(e4)
        d4=self.d4(torch.cat([self.u4(b),  e3], 1))
        d3=self.d3(torch.cat([self.u3(d4), e2], 1))
        d2=self.d2(torch.cat([self.u2(d3), e1], 1))
        d1=self.d1(torch.cat([self.u1(d2), inp], 1))
        return self.out(d1)

class cGANDiscriminator(nn.Module):
    def __init__(self, in_ch=1, cond_ch=4, bf=32):
        super().__init__()
        ti = in_ch + cond_ch
        self.model = nn.Sequential(
            spectral_norm(nn.Conv2d(ti,   bf,   4, 2, 1)), nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(bf,   bf*2, 4, 2, 1)), nn.BatchNorm2d(bf*2), nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(bf*2, bf*4, 4, 2, 1)), nn.BatchNorm2d(bf*4), nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(bf*4, bf*4, 4, 1, 1)), nn.BatchNorm2d(bf*4), nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(bf*4, 1,    4, 1, 1)),
        )
    def forward(self, img, cond): return self.model(torch.cat([img, cond], 1))

G = cGANGenerator(1, 4, 1, 32).to(device)
D = cGANDiscriminator(1, 4, 32).to(device)
print(f"G: {sum(p.numel() for p in G.parameters()):,} params")
print(f"D: {sum(p.numel() for p in D.parameters()):,} params")

G: 526,043 params
D: 431,585 params


In [24]:
# ══════════════════════════════════════════════════════════════
#   cGAN Training — Complete Single Cell
#   Runs all stages/epochs, saves per-epoch:
#     • sample_01..05.png  (4-panel: Real|Generated|Diff|Mask)
#     • weights.pt         (G, D, optimizers, loss history)
#   Plus loss_curve_latest.png and checkpoints every 10 epochs
# ══════════════════════════════════════════════════════════════

import os, sys, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── Directories ───────────────────────────────────────────────
SAVE_DIR = r"F:\shn\Conditional GAN"
IMG_DIR  = os.path.join(SAVE_DIR, "epoch_samples")
CKPT_DIR = os.path.join(SAVE_DIR, "checkpoints")
os.makedirs(IMG_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────
LAM_L1                = 10.0
LAM_SSIM              = 5.0
PRINT_EVERY_N_BATCHES = 10
N_SAMPLES             = 5

# ══════════════════════════════════════════════════════════════
# Helper Functions
# ══════════════════════════════════════════════════════════════

def bce_loss(pred, target, smooth=True):
    """Accepts bool (label-smoothed) or pre-built tensor."""
    if isinstance(target, bool):
        val    = 0.9 if (target and smooth) else (0.1 if (not target and smooth) else float(target))
        target = torch.full_like(pred, val)
    return F.binary_cross_entropy_with_logits(pred, target)


def save_epoch_samples(G, val_ds, device, sc, stage_idx, epoch_num,
                       img_dir, gl, dl_list, n=5):
    """Save n individual 4-panel PNGs into epoch folder. Returns folder path."""
    G.eval()

    epoch_folder = os.path.join(img_dir,
                                f"stage_{stage_idx+1}",
                                f"epoch_{epoch_num:04d}")
    os.makedirs(epoch_folder, exist_ok=True)

    loader        = DataLoader(val_ds, batch_size=n, shuffle=True, num_workers=0)
    real_b, msk_b = next(iter(loader))
    real_b        = sc.resize(real_b.to(device))
    msk_b         = sc.resize(msk_b.to(device))

    with torch.no_grad():
        fake_b = G(real_b, msk_b)

    def t2np(t):
        return t.squeeze(1).clamp(0, 1).cpu().numpy()

    real_np = t2np(real_b)
    fake_np = t2np(fake_b)
    diff_np = np.abs(real_np - fake_np)
    roi_np  = t2np(msk_b.max(dim=1, keepdim=True).values)

    saved = []
    for i in range(min(n, real_np.shape[0])):
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        fig.patch.set_facecolor('#1a1a2e')
        fig.suptitle(
            f"Stage {stage_idx+1}  |  Epoch {epoch_num:04d}  |  Sample {i+1}/{n}\n"
            f"G Loss: {gl[-1]:.6f}    D Loss: {dl_list[-1]:.6f}",
            fontsize=11, fontweight='bold', color='white', y=1.02
        )
        for ax, (img, cmap, title) in zip(axes, [
            (real_np[i], "gray",   "Real Input"),
            (fake_np[i], "gray",   "Generated"),
            (diff_np[i], "hot",    "Abs Diff"),
            (roi_np[i],  "plasma", "ROI Mask"),
        ]):
            ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
            ax.set_title(title, fontsize=11, fontweight='bold', color='white', pad=6)
            ax.axis("off")
            for spine in ax.spines.values():
                spine.set_edgecolor('white')
                spine.set_linewidth(1.2)

        plt.tight_layout(pad=1.5)
        out = os.path.join(epoch_folder, f"sample_{i+1:02d}.png")
        plt.savefig(out, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
        plt.close(fig)
        saved.append(out)

    G.train()
    return epoch_folder, saved


def save_weights(folder, G, D, oG, oD, stage_idx, ep, gl, dl_list):
    """Save full training state into epoch folder as weights.pt."""
    path = os.path.join(folder, "weights.pt")
    torch.save({
        "G":      G.state_dict(),
        "D":      D.state_dict(),
        "oG":     oG.state_dict(),
        "oD":     oD.state_dict(),
        "stage":  stage_idx,
        "epoch":  ep,
        "g_loss": gl[-1],
        "d_loss": dl_list[-1],
        "lr":     oG.param_groups[0]['lr'],
        "gl":     gl,
        "dl":     dl_list,
    }, path)
    return path


def save_loss_curve(gl, dl_list, save_dir, stage_idx, epoch_num):
    """Save loss curve PNG to save_dir root."""
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')
    ax.plot(gl,      label="Generator",     color="#4fc3f7", lw=1.8)
    ax.plot(dl_list, label="Discriminator", color="#ef5350", lw=1.8)
    boundary = 0
    for s in range(stage_idx):
        boundary += PSched.STAGES[s]["epochs"]
        if boundary < len(gl):
            ax.axvline(boundary, color="yellow", ls="--", alpha=0.5, label=f"Stage {s+2}")
    ax.set_title(f"Loss Curve — Stage {stage_idx+1} | Epoch {epoch_num}",
                 color='white', fontsize=12)
    ax.set_xlabel("Epoch", color='white')
    ax.set_ylabel("Loss",  color='white')
    ax.tick_params(colors='white')
    ax.legend(facecolor='#16213e', labelcolor='white')
    ax.grid(alpha=0.2, color='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#4fc3f7')
    plt.tight_layout()
    path = os.path.join(save_dir, "loss_curve_latest.png")
    plt.savefig(path, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    return path


def _build_stage(sc, oG, oD, train_ds, device):
    """Create DataLoader + LR schedulers for current stage."""
    sc.set_lr(oG, oD)
    tdl = DataLoader(
        train_ds, batch_size=sc.batch, shuffle=True,
        num_workers=0, pin_memory=(device.type == "cuda"),
        persistent_workers=False,
    )
    sg = CosineAnnealingLR(oG, T_max=sc.epochs, eta_min=sc.lr * 0.1)
    sd = CosineAnnealingLR(oD, T_max=sc.epochs, eta_min=sc.lr * 0.1)
    return tdl, sg, sd

# ══════════════════════════════════════════════════════════════
# Optimizers & State
# ══════════════════════════════════════════════════════════════

oG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
oD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

sc               = PSched()
gl               = []
dl_list          = []
stage_idx        = 0
ep               = 0
total_epochs_all = sum(s["epochs"] for s in PSched.STAGES)

train_dl, sched_G, sched_D = _build_stage(sc, oG, oD, train_ds, device)

print(f"\n{'═'*60}")
print(f"  cGAN Training — {sc.n} Stages | {total_epochs_all} Total Epochs")
print(f"{'═'*60}")
print(f"  Samples     →  {IMG_DIR}")
print(f"  Checkpoints →  {CKPT_DIR}")
print(f"{'═'*60}\n")

# ══════════════════════════════════════════════════════════════
# Training Loop — All Stages, All Epochs
# ══════════════════════════════════════════════════════════════

while stage_idx < sc.n:

    print(f"\n{'═'*60}")
    print(f"  STAGE {stage_idx+1}/{sc.n}  "
          f"res={sc.res}  batch={sc.batch}  "
          f"lr={sc.lr:.1e}  epochs={sc.epochs}")
    print(f"{'═'*60}\n")

    n_batches = len(train_dl)

    while ep < sc.epochs:

        G.train(); D.train()
        epoch_g = epoch_d = 0.0
        t0 = time.time()

        done  = sum(s["epochs"] for s in PSched.STAGES[:stage_idx]) + ep
        e_bar = "█" * int(20 * ep / sc.epochs)           + "░" * (20 - int(20 * ep / sc.epochs))
        t_bar = "█" * int(40 * done / total_epochs_all)  + "░" * (40 - int(40 * done / total_epochs_all))

        print(f"  ▶ Epoch {ep+1:3d}/{sc.epochs}  [{e_bar}]")
        print(f"    Overall [{t_bar}] {done}/{total_epochs_all}")
        print(f"  {'─'*56}")

        # ── Batch loop ───────────────────────────────────────
        for batch_idx, (real, masks) in enumerate(train_dl):
            real = sc.resize(real.to(device, non_blocking=True))
            cond = sc.resize(masks.to(device, non_blocking=True))

            # D step
            with torch.no_grad():
                fake = G(real, cond)
            d_real = D(real, cond)
            d_fake = D(fake.detach(), cond)
            loss_D = 0.5 * (bce_loss(d_real, True) + bce_loss(d_fake, False))
            oD.zero_grad(set_to_none=True)
            loss_D.backward()
            nn.utils.clip_grad_norm_(D.parameters(), 1.0)
            oD.step()

            # G step
            fake         = G(real, cond)
            d_fake_for_g = D(fake, cond)
            loss_G       = (bce_loss(d_fake_for_g, True)
                            + LAM_L1   * roi_l1(fake, real, cond)
                            + LAM_SSIM * ssim_loss(fake, real))
            oG.zero_grad(set_to_none=True)
            loss_G.backward()
            nn.utils.clip_grad_norm_(G.parameters(), 1.0)
            oG.step()

            epoch_g += loss_G.item()
            epoch_d += loss_D.item()

            if (batch_idx+1) % PRINT_EVERY_N_BATCHES == 0 or (batch_idx+1) == n_batches:
                avg_g    = epoch_g / (batch_idx + 1)
                avg_d    = epoch_d / (batch_idx + 1)
                pct      = (batch_idx + 1) / n_batches * 100
                mini_bar = "▓" * int(pct / 5) + "░" * (20 - int(pct / 5))
                print(f"    [{mini_bar}] {pct:5.1f}%  "
                      f"b {batch_idx+1:3d}/{n_batches}  "
                      f"G={avg_g:.4f}  D={avg_d:.4f}  "
                      f"t={time.time()-t0:.1f}s", flush=True)

        # ── Epoch bookkeeping ────────────────────────────────
        gl.append(epoch_g / n_batches)
        dl_list.append(epoch_d / n_batches)
        sched_G.step()
        sched_D.step()
        ep += 1

        elapsed = time.time() - t0
        print(f"\n  ✅ Epoch {ep:04d}/{sc.epochs} complete")
        print(f"     G Loss : {gl[-1]:.6f}")
        print(f"     D Loss : {dl_list[-1]:.6f}")
        print(f"     LR     : {oG.param_groups[0]['lr']:.2e}")
        print(f"     Time   : {elapsed:.1f}s")

        # ── Save 5 sample images into epoch folder ───────────
        folder, paths = save_epoch_samples(
            G, val_ds, device, sc, stage_idx, ep, IMG_DIR, gl, dl_list, N_SAMPLES
        )
        print(f"\n     🖼  {folder}")
        for p in paths:
            print(f"          📷 {os.path.basename(p)}")

        # ── Save weights.pt into same epoch folder ────────────
        w_path = save_weights(folder, G, D, oG, oD, stage_idx, ep, gl, dl_list)
        print(f"          💾 weights.pt")

        # ── Save loss curve ───────────────────────────────────
        curve = save_loss_curve(gl, dl_list, SAVE_DIR, stage_idx, ep)
        print(f"     📈 {curve}")

        # ── Checkpoint every 10 epochs in CKPT_DIR ────────────
        if ep % 10 == 0:
            ckpt = os.path.join(CKPT_DIR, f"ckpt_s{stage_idx+1}_e{ep:04d}.pt")
            torch.save({
                "G": G.state_dict(), "D": D.state_dict(),
                "oG": oG.state_dict(), "oD": oD.state_dict(),
                "stage": stage_idx, "epoch": ep,
                "gl": gl, "dl": dl_list,
            }, ckpt)
            print(f"     💾 {ckpt}")

        print(f"\n  {'─'*56}\n")

    # ── Stage complete → advance ──────────────────────────────
    print(f"\n🏁 Stage {stage_idx+1} COMPLETE!")
    stage_idx += 1
    ep = 0

    if stage_idx < sc.n:
        sc.advance()
        train_dl, sched_G, sched_D = _build_stage(sc, oG, oD, train_ds, device)
        print(f"  ➡  Stage {stage_idx+1}: "
              f"res={sc.res}  batch={sc.batch}  lr={sc.lr:.1e}\n")

# ── Final model save ─────────────────────────────────────────
final_path = os.path.join(SAVE_DIR, "generator_final.pt")
torch.save(G.state_dict(), final_path)

print(f"\n{'═'*60}")
print(f"  🎉 ALL TRAINING COMPLETE!")
print(f"  Final model → {final_path}")
print(f"  Samples     → {IMG_DIR}")
print(f"  Checkpoints → {CKPT_DIR}")
print(f"{'═'*60}")


════════════════════════════════════════════════════════════
  cGAN Training — 3 Stages | 200 Total Epochs
════════════════════════════════════════════════════════════
  Samples     →  F:\shn\Conditional GAN\epoch_samples
  Checkpoints →  F:\shn\Conditional GAN\checkpoints
════════════════════════════════════════════════════════════


════════════════════════════════════════════════════════════
  STAGE 1/3  res=32  batch=512  lr=2.0e-04  epochs=40
════════════════════════════════════════════════════════════

  ▶ Epoch   1/40  [░░░░░░░░░░░░░░░░░░░░]
    Overall [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 0/200
  ────────────────────────────────────────────────────────
    [▓▓▓▓▓▓▓▓▓▓▓░░░░░░░░░]  58.8%  b  10/17  G=19.9852  D=0.4647  t=29.9s
    [▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓] 100.0%  b  17/17  G=19.7142  D=0.4101  t=43.9s

  ✅ Epoch 0001/40 complete
     G Loss : 19.714229
     D Loss : 0.410128
     LR     : 2.00e-04
     Time   : 43.9s

     🖼  F:\shn\Conditional GAN\epoch_samples\stage_1\epoc

In [29]:
# ══════════════ Cell 7: Loss Curves ══════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(gl, label="Generator",     color="steelblue", lw=1.5)
ax.plot(dl_list, label="Discriminator", color="tomato",    lw=1.5)
for b in [22, 60]:
    if b < len(gl):
        ax.axvline(b, color="gray", linestyle="--", alpha=0.5)
ax.set_title("cGAN Training Losses (Full)"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
w = min(50, len(gl))
ax2.plot(gl[-w:], label="Generator",     color="steelblue", lw=1.5)
ax2.plot(dl_list[-w:], label="Discriminator", color="tomato",    lw=1.5)
ax2.set_title(f"Last {w} Epochs"); ax2.set_xlabel("Epoch")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {SAVE_DIR}/loss_curves.png")

Saved → F:\shn\Conditional GAN/loss_curves.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_57540\4237045863.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
# ══════════════ Cell 8: Visual Sanity Check ══════════════

G.eval()
val_dl      = DataLoader(val_ds, batch_size=4, shuffle=True, num_workers=0)
real_b, msk_b = next(iter(val_dl))
real_b = real_b.to(device); msk_b = msk_b.to(device)

with torch.no_grad():
    fake_b = G(real_b, msk_b)

def to_np(t): return t.squeeze(1).cpu().numpy()

real_np = to_np(real_b); fake_np = to_np(fake_b)
roi_np  = to_np(composite_roi(msk_b))
diff_np = np.abs(real_np - fake_np)

n = real_np.shape[0]
fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
for idx in range(n):
    axes[idx,0].imshow(real_np[idx],  cmap="gray");   axes[idx,0].set_title("Real")
    axes[idx,1].imshow(fake_np[idx],  cmap="gray");   axes[idx,1].set_title("Generated")
    axes[idx,2].imshow(diff_np[idx],  cmap="hot");    axes[idx,2].set_title("Abs Diff")
    axes[idx,3].imshow(roi_np[idx],   cmap="plasma"); axes[idx,3].set_title("ROI Mask")
    for ax in axes[idx]: ax.axis("off")

plt.suptitle("cGAN [FIXED v2] — Real vs Generated", fontsize=14)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/visual_check.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {SAVE_DIR}/visual_check.png")

Saved → F:\shn\Conditional GAN/visual_check.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_57540\2820537320.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
# ══════════════ Cell 9: Comprehensive Evaluation ══════════════

def compute_metrics(G, dataset, device, n_batches=26, batch_size=8):
    G.eval()
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    roi_mse_list, l1_list, roi_ssim_list, dsc_list, kl_list = [], [], [], [], []

    with torch.no_grad():
        for i, (real, masks) in enumerate(tqdm(dl, desc="Evaluating cGAN")):
            if i >= n_batches: break
            real = real.to(device); cond = masks.to(device)
            fake = G(real, cond)
            roi  = composite_roi(cond)

            roi_mse_list.append((((fake - real)**2 * roi).sum() / (roi.sum() + 1e-8)).item())
            l1_list.append(F.l1_loss(fake, real).item())

            if USE_SSIM:
                roi_ssim_list.append(ssim_fn(fake.clamp(0,1), real.clamp(0,1), data_range=1.0, size_average=True).item())
            else:
                roi_ssim_list.append(float('nan'))

            pred_bin = (fake > 0.5).float(); real_bin = (real > 0.5).float()
            dsc_list.append(((2*(pred_bin*real_bin).sum()) / (pred_bin.sum()+real_bin.sum()+1e-8)).item())

            p = torch.histc(fake.cpu(), bins=256, min=0, max=1) + 1e-8
            q = torch.histc(real.cpu(), bins=256, min=0, max=1) + 1e-8
            p /= p.sum(); q /= q.sum()
            kl_list.append((p * (p/q).log()).sum().item())

    return {"ROI-MSE": np.mean(roi_mse_list), "L1 Loss": np.mean(l1_list),
            "ROI-SSIM": np.mean(roi_ssim_list), "DSC": np.mean(dsc_list), "KL-Div": np.mean(kl_list)}

metrics  = compute_metrics(G, val_ds, device)
BASELINE = {"ROI-MSE": 0.092034, "L1 Loss": 0.199110, "ROI-SSIM": 0.312798, "DSC": 0.661137, "KL-Div": 6.905633}
BETTER   = {"ROI-MSE": lambda n,b: n<b, "L1 Loss": lambda n,b: n<b,
            "ROI-SSIM": lambda n,b: n>b, "DSC": lambda n,b: n>b, "KL-Div": lambda n,b: n<b}
DIR      = {"ROI-MSE": "↓", "L1 Loss": "↓", "ROI-SSIM": "↑", "DSC": "↑", "KL-Div": "↓"}

print("\n" + "═"*68)
print("  CONDITIONAL GAN [FIXED v2] — EVALUATION SUMMARY")
print("═"*68)
print(f"  {'Metric':<12} {'Dir':<4} {'Baseline':>12} {'Fixed v2':>12} {'Δ':>10}")
print("  " + "-"*64)
for metric, val in metrics.items():
    base = BASELINE[metric]
    delta = val - base
    sym = "✅" if BETTER[metric](val, base) else "❌"
    print(f"  {metric:<12} ({DIR[metric]})  {base:>12.6f} {val:>12.6f} {delta:>+10.6f} {sym}")
print("═"*68)

Evaluating cGAN:   0%|          | 0/208 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════════════
  CONDITIONAL GAN [FIXED v2] — EVALUATION SUMMARY
════════════════════════════════════════════════════════════════════
  Metric       Dir      Baseline     Fixed v2          Δ
  ----------------------------------------------------------------
  ROI-MSE      (↓)      0.092034     0.000308  -0.091726 ✅
  L1 Loss      (↓)      0.199110     0.008498  -0.190612 ✅
  ROI-SSIM     (↑)      0.312798     0.979221  +0.666423 ✅
  DSC          (↑)      0.661137     0.989963  +0.328826 ✅
  KL-Div       (↓)      6.905633     1.589638  -5.315995 ✅
════════════════════════════════════════════════════════════════════


In [34]:
import torch
import os

# 1. Path to your latest checkpoint
checkpoint_path = r'F:\shn\Conditional GAN\checkpoints\ckpt_s3_e0040.pt'

def calculate_dice(predict, target, smooth=1e-6):
    """Calculates the Dice Similarity Coefficient (DSC)"""
    # Threshold the predictions to get a binary mask
    predict = (predict > 0.5).float()
    target = (target > 0.5).float()
    
    intersection = (predict * target).sum()
    dice = (2. * intersection + smooth) / (predict.sum() + target.sum() + smooth)
    return dice.item()

# 2. Loading the file
if os.path.exists(checkpoint_path):
    # Use torch.load for .pt files
    checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))
    
    # Check if weights are nested (common in GANs)
    # It might be under 'model_state_dict', 'G_state_dict', etc.
    if isinstance(checkpoint, dict):
        print("Keys in checkpoint:", checkpoint.keys())
        state_dict = checkpoint.get('state_dict', checkpoint)
    else:
        state_dict = checkpoint

    print("Successfully loaded Stage 3, Epoch 40.")
else:
    print("File not found. Check the path and file name.")

# 3. Next Step: 
# Load 'state_dict' into your model: model.load_state_dict(state_dict)

Keys in checkpoint: dict_keys(['G', 'D', 'oG', 'oD', 'stage', 'epoch', 'gl', 'dl'])
Successfully loaded Stage 3, Epoch 40.
